# Exp 012: Perch Temporal Light

Simplified first-pass reproduction of the strong `0.924` ProtoSSM direction. The goal is to measure how much gain comes from **file-level temporal modeling over Perch embeddings/logits** without bringing in the full residual stack or heavy leaderboard-oriented postprocessing.


## Hypothesis

If the main strength of the `0.924` reference comes from temporal modeling over `Perch` features, then a lighter first-pass model on the trusted `59` fully labeled soundscape files should already beat the older `exp_003` downstream branch on honest grouped OOF.


In [1]:
from __future__ import annotations

import json
import math
import os
import random
import re
from dataclasses import asdict, dataclass
from pathlib import Path
import typing as tp

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm


In [2]:
ROOT = Path('/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026')
DATA = ROOT / 'data' / 'birdclef-2026'
PERCH_META_DIR = ROOT / 'data' / 'perch_meta'
EXPERIMENTS = ROOT / 'experiments'
OUTPUT_DIR = EXPERIMENTS / 'outputs' / 'exp_012_perch_temporal_light'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    experiment_id: str = 'exp_012'
    experiment_name: str = 'perch_temporal_light'
    random_seed: int = 1086
    n_splits: int = 3

    d_model: int = 192
    d_state: int = 16
    n_ssm_layers: int = 2
    dropout: float = 0.15
    meta_dim: int = 16

    n_epochs: int = 45
    lr: float = 1e-3
    weight_decay: float = 2e-3
    patience: int = 10
    label_smoothing: float = 0.02
    distill_weight: float = 0.10
    family_weight: float = 0.10
    pos_weight_cap: float = 30.0

    save_oof_outputs: bool = True


CFG = Config()
RUN_OOF = True
RUN_FINAL_TRAIN = False
RESUME_FINAL_TRAIN = True


In [3]:
def set_random_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


set_random_seed(CFG.random_seed)
device = pick_device()
print({
    'root': str(ROOT),
    'perch_meta_dir_exists': PERCH_META_DIR.exists(),
    'device': str(device),
    'output_dir': str(OUTPUT_DIR),
})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'perch_meta_dir_exists': True, 'device': 'mps', 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_012_perch_temporal_light'}


In [4]:
taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA / 'train_soundscapes_labels.csv').drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}


def parse_label_tokens(x: str) -> list[str]:
    if pd.isna(x):
        return []
    return [token.strip() for token in str(x).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    labels = sorted(set(lbl for value in series for lbl in parse_label_tokens(value) if lbl in label_to_idx))
    return labels


FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_soundscape_filename(name: str) -> dict[str, tp.Any]:
    match = FNAME_RE.match(name)
    if not match:
        return {'site': None, 'hour_utc': -1, 'file_id': None}
    file_id, site, ymd, hms = match.groups()
    return {'site': site, 'hour_utc': int(hms[:2]), 'file_id': file_id}


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for row_idx, labels in enumerate(sc_clean['label_list']):
    for label in labels:
        if label in label_to_idx:
            Y_SC[row_idx, label_to_idx[label]] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)

print({
    'full_files': len(full_files),
    'trusted_windows': len(full_truth),
    'active_classes_in_trusted_set': int((Y_SC[full_truth['index'].to_numpy()].sum(axis=0) > 0).sum()),
})


{'full_files': 59, 'trusted_windows': 708, 'active_classes_in_trusted_set': 71}


In [5]:
meta_full = pd.read_parquet(PERCH_META_DIR / 'full_perch_meta.parquet')
arr = np.load(PERCH_META_DIR / 'full_perch_arrays.npz')
scores_full_raw = arr['scores_full_raw'].astype(np.float32)
emb_full = arr['emb_full'].astype(np.float32)

full_truth_aligned = full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index()
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()].astype(np.float32)

assert np.all(full_truth_aligned['filename'].values == meta_full['filename'].values)
assert np.all(full_truth_aligned['row_id'].values == meta_full['row_id'].values)

print({
    'meta_full_shape': tuple(meta_full.shape),
    'scores_full_raw_shape': tuple(scores_full_raw.shape),
    'emb_full_shape': tuple(emb_full.shape),
    'y_full_shape': tuple(Y_FULL.shape),
})


{'meta_full_shape': (708, 4), 'scores_full_raw_shape': (708, 234), 'emb_full_shape': (708, 1536), 'y_full_shape': (708, 234)}


In [6]:
def macro_auc_skip_empty(y_true: np.ndarray, y_score: np.ndarray) -> float:
    positive_mask = y_true.sum(axis=0) > 0
    negative_mask = (y_true.shape[0] - y_true.sum(axis=0)) > 0
    scored_mask = positive_mask & negative_mask
    if not scored_mask.any():
        return float('nan')
    return float(roc_auc_score(y_true[:, scored_mask], y_score[:, scored_mask], average='macro'))


def reshape_to_files(flat_array: np.ndarray, meta_df: pd.DataFrame, n_windows: int = 12):
    filenames = meta_df['filename'].to_numpy()
    unique_files = []
    seen = set()
    for filename in filenames:
        if filename not in seen:
            unique_files.append(filename)
            seen.add(filename)
    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows, (len(flat_array), n_files, n_windows)
    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files


def build_site_mapping(meta_df: pd.DataFrame):
    sites = sorted(meta_df['site'].dropna().astype(str).unique().tolist())
    site_to_idx = {site: idx + 1 for idx, site in enumerate(sites)}
    return site_to_idx, len(sites) + 1


def get_file_metadata(meta_df: pd.DataFrame, file_list: list[str], site_to_idx: dict[str, int], n_sites_max: int):
    file_to_row = {}
    filenames = meta_df['filename'].to_numpy()
    sites = meta_df['site'].astype(str).to_numpy()
    hours = meta_df['hour_utc'].astype(int).to_numpy()
    for idx, filename in enumerate(filenames):
        if filename not in file_to_row:
            file_to_row[filename] = idx

    site_ids = np.zeros(len(file_list), dtype=np.int64)
    hour_ids = np.zeros(len(file_list), dtype=np.int64)
    group_ids = np.empty(len(file_list), dtype=object)
    for file_idx, filename in enumerate(file_list):
        row = file_to_row[filename]
        site = sites[row]
        hour = hours[row]
        site_ids[file_idx] = min(site_to_idx.get(site, 0), n_sites_max - 1)
        hour_ids[file_idx] = hour % 24
        group_ids[file_idx] = site
    return site_ids, hour_ids, group_ids


def build_taxonomy_groups(taxonomy_df: pd.DataFrame, primary_labels: list[str]):
    for column in ['family', 'order', 'class_name']:
        if column in taxonomy_df.columns:
            group_map = taxonomy_df.set_index('primary_label')[column].astype(str).to_dict()
            break
    else:
        group_map = {label: 'Unknown' for label in primary_labels}

    groups = sorted(set(group_map.values()))
    group_to_idx = {group: idx for idx, group in enumerate(groups)}
    class_to_group = [group_to_idx[group_map.get(label, 'Unknown')] for label in primary_labels]
    return len(groups), class_to_group


emb_files, file_list = reshape_to_files(emb_full, meta_full)
logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
labels_files, _ = reshape_to_files(Y_FULL, meta_full)
site_to_idx, n_sites = build_site_mapping(meta_full)
site_ids_all, hours_all, group_ids_all = get_file_metadata(meta_full, file_list, site_to_idx, n_sites)
n_families, class_to_family = build_taxonomy_groups(taxonomy, PRIMARY_LABELS)

file_families = np.zeros((len(file_list), n_families), dtype=np.float32)
for file_idx in range(len(file_list)):
    active_classes = np.where(labels_files[file_idx].sum(axis=0) > 0)[0]
    for class_idx in active_classes:
        file_families[file_idx, class_to_family[class_idx]] = 1.0

raw_perch_auc = macro_auc_skip_empty(Y_FULL, scores_full_raw)
legacy_probe_snapshot = ROOT / 'experiments' / 'outputs' / 'exp_003_perch_downstream_reproduction' / 'result_snapshot.json'
legacy_probe_auc = None
if legacy_probe_snapshot.exists():
    legacy_probe_auc = json.loads(legacy_probe_snapshot.read_text()).get('best_macro_auc')

print({
    'emb_files_shape': tuple(emb_files.shape),
    'logits_files_shape': tuple(logits_files.shape),
    'labels_files_shape': tuple(labels_files.shape),
    'n_sites': int(n_sites),
    'raw_perch_auc': raw_perch_auc,
    'legacy_exp003_best_auc': legacy_probe_auc,
})


{'emb_files_shape': (59, 12, 1536), 'logits_files_shape': (59, 12, 234), 'labels_files_shape': (59, 12, 234), 'n_sites': 9, 'raw_perch_auc': 0.7390178442294307, 'legacy_exp003_best_auc': None}


In [7]:
class SelectiveSSM(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, steps, d_model = x.shape
        xz = self.in_proj(x)
        x_ssm, _ = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :steps].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(batch_size, d_model, self.d_state, device=x.device)
        outputs = []
        for step in range(steps):
            dt_t = dt[:, step, :]
            dA = torch.exp(A[None, :, :] * dt_t[:, :, None])
            dB = dt_t[:, :, None] * B[:, step, None, :]
            h = h * dA + x[:, step, :, None] * dB
            y_t = (h * C[:, step, None, :]).sum(-1)
            outputs.append(y_t)
        y = torch.stack(outputs, dim=1)
        return y + x * self.D[None, None, :]


class ProtoTemporalLight(nn.Module):
    def __init__(self, d_input: int = 1536, n_classes: int = N_CLASSES, n_windows: int = 12):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, CFG.d_model),
            nn.LayerNorm(CFG.d_model),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, CFG.d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, CFG.meta_dim)
        self.hour_emb = nn.Embedding(24, CFG.meta_dim)
        self.meta_proj = nn.Linear(2 * CFG.meta_dim, CFG.d_model)

        self.ssm_fwd = nn.ModuleList([SelectiveSSM(CFG.d_model, CFG.d_state) for _ in range(CFG.n_ssm_layers)])
        self.ssm_bwd = nn.ModuleList([SelectiveSSM(CFG.d_model, CFG.d_state) for _ in range(CFG.n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * CFG.d_model, CFG.d_model) for _ in range(CFG.n_ssm_layers)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(CFG.d_model) for _ in range(CFG.n_ssm_layers)])
        self.ssm_drop = nn.Dropout(CFG.dropout)

        self.prototypes = nn.Parameter(torch.randn(n_classes, CFG.d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
        self.family_head = nn.Linear(CFG.d_model, n_families)

    def init_prototypes_from_data(self, emb_flat: torch.Tensor, labels_flat: torch.Tensor) -> None:
        with torch.no_grad():
            h = self.input_proj(emb_flat)
            for class_idx in range(self.n_classes):
                mask = labels_flat[:, class_idx] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[class_idx] = F.normalize(h[mask].mean(dim=0), dim=0)

    def forward(self, emb: torch.Tensor, perch_logits: torch.Tensor, site_ids: torch.Tensor, hours: torch.Tensor):
        h = self.input_proj(emb)
        h = h + self.pos_enc[:, :h.shape[1], :]
        meta = self.meta_proj(torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
        h = h + meta[:, None, :]

        for fwd, bwd, merge, norm in zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm):
            residual = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)

        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        proto_logits = torch.matmul(h_norm, p_norm.T) * F.softplus(self.proto_temp) + self.class_bias[None, None, :]
        alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
        species_logits = alpha * proto_logits + (1.0 - alpha) * perch_logits
        family_logits = self.family_head(h.mean(dim=1))
        return species_logits, family_logits, h

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def make_pos_weight(labels_tr: torch.Tensor) -> torch.Tensor:
    pos_counts = labels_tr.sum(dim=(0, 1))
    total = labels_tr.shape[0] * labels_tr.shape[1]
    pos_weight = (total - pos_counts) / (pos_counts + 1.0)
    return pos_weight.clamp(max=CFG.pos_weight_cap)


In [8]:
def smooth_labels(labels: np.ndarray, amount: float) -> np.ndarray:
    if amount <= 0:
        return labels.astype(np.float32, copy=True)
    return (labels * (1.0 - amount) + amount / 2.0).astype(np.float32, copy=False)


def train_single_fold(
    emb_train: np.ndarray,
    logits_train: np.ndarray,
    labels_train: np.ndarray,
    site_train: np.ndarray,
    hour_train: np.ndarray,
    fam_train: np.ndarray,
    emb_valid: np.ndarray,
    logits_valid: np.ndarray,
    labels_valid: np.ndarray,
    site_valid: np.ndarray,
    hour_valid: np.ndarray,
    fam_valid: np.ndarray,
) -> tuple[ProtoTemporalLight, dict[str, list[float]]]:
    model = ProtoTemporalLight(d_input=emb_train.shape[-1]).to(device)
    # Prototype initialization must run on the same device as the model.
    flat_emb = torch.tensor(emb_train.reshape(-1, emb_train.shape[-1]), dtype=torch.float32, device=device)
    flat_labels = torch.tensor(labels_train.reshape(-1, labels_train.shape[-1]), dtype=torch.float32, device=device)
    model.init_prototypes_from_data(flat_emb, flat_labels)

    labels_train_sm = smooth_labels(labels_train, CFG.label_smoothing)
    labels_valid_sm = labels_valid.astype(np.float32, copy=False)

    emb_tr = torch.tensor(emb_train, dtype=torch.float32, device=device)
    logits_tr = torch.tensor(logits_train, dtype=torch.float32, device=device)
    labels_tr = torch.tensor(labels_train_sm, dtype=torch.float32, device=device)
    site_tr = torch.tensor(site_train, dtype=torch.long, device=device)
    hour_tr = torch.tensor(hour_train, dtype=torch.long, device=device)
    fam_tr = torch.tensor(fam_train, dtype=torch.float32, device=device)

    emb_va = torch.tensor(emb_valid, dtype=torch.float32, device=device)
    logits_va = torch.tensor(logits_valid, dtype=torch.float32, device=device)
    labels_va = torch.tensor(labels_valid_sm, dtype=torch.float32, device=device)
    site_va = torch.tensor(site_valid, dtype=torch.long, device=device)
    hour_va = torch.tensor(hour_valid, dtype=torch.long, device=device)
    fam_va = torch.tensor(fam_valid, dtype=torch.float32, device=device)

    pos_weight = make_pos_weight(labels_tr)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG.lr,
        epochs=CFG.n_epochs,
        steps_per_epoch=1,
        pct_start=0.1,
        anneal_strategy='cos',
    )

    best_state = None
    best_val_auc = -float('inf')
    wait = 0
    history = {'train_loss': [], 'valid_loss': [], 'valid_auc': []}

    for epoch in range(1, CFG.n_epochs + 1):
        model.train()
        species_out, family_out, _ = model(emb_tr, logits_tr, site_tr, hour_tr)
        loss_main = F.binary_cross_entropy_with_logits(species_out, labels_tr, pos_weight=pos_weight[None, None, :])
        loss_distill = F.mse_loss(species_out, logits_tr)
        loss_family = F.binary_cross_entropy_with_logits(family_out, fam_tr)
        loss = loss_main + CFG.distill_weight * loss_distill + CFG.family_weight * loss_family

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            valid_species_out, valid_family_out, _ = model(emb_va, logits_va, site_va, hour_va)
            valid_loss = F.binary_cross_entropy_with_logits(valid_species_out, labels_va, pos_weight=pos_weight[None, None, :])
            valid_loss = valid_loss + CFG.family_weight * F.binary_cross_entropy_with_logits(valid_family_out, fam_va)
            valid_probs = torch.sigmoid(valid_species_out).detach().cpu().numpy().reshape(-1, N_CLASSES)
        valid_true = labels_valid.reshape(-1, N_CLASSES)
        valid_auc = macro_auc_skip_empty(valid_true, valid_probs)

        history['train_loss'].append(float(loss.item()))
        history['valid_loss'].append(float(valid_loss.item()))
        history['valid_auc'].append(float(valid_auc))

        if valid_auc > best_val_auc:
            best_val_auc = float(valid_auc)
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        print({'epoch': epoch, 'train_loss': float(loss.item()), 'valid_loss': float(valid_loss.item()), 'valid_auc': float(valid_auc)})
        if wait >= CFG.patience:
            print(f'Early stop at epoch {epoch}')
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    return model, history


In [9]:
def run_grouped_oof() -> dict[str, tp.Any]:
    splitter = GroupKFold(n_splits=CFG.n_splits)
    oof_logits = np.zeros_like(labels_files, dtype=np.float32)
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(emb_files, groups=group_ids_all), 1):
        print(f'\n=== Fold {fold} ===')
        model, history = train_single_fold(
            emb_train=emb_files[train_idx],
            logits_train=logits_files[train_idx],
            labels_train=labels_files[train_idx],
            site_train=site_ids_all[train_idx],
            hour_train=hours_all[train_idx],
            fam_train=file_families[train_idx],
            emb_valid=emb_files[valid_idx],
            logits_valid=logits_files[valid_idx],
            labels_valid=labels_files[valid_idx],
            site_valid=site_ids_all[valid_idx],
            hour_valid=hours_all[valid_idx],
            fam_valid=file_families[valid_idx],
        )
        model.eval()
        with torch.no_grad():
            emb_va = torch.tensor(emb_files[valid_idx], dtype=torch.float32, device=device)
            logits_va = torch.tensor(logits_files[valid_idx], dtype=torch.float32, device=device)
            site_va = torch.tensor(site_ids_all[valid_idx], dtype=torch.long, device=device)
            hour_va = torch.tensor(hours_all[valid_idx], dtype=torch.long, device=device)
            species_out, _, _ = model(emb_va, logits_va, site_va, hour_va)
            probs = torch.sigmoid(species_out).detach().cpu().numpy().astype(np.float32)
        oof_logits[valid_idx] = probs
        fold_auc = macro_auc_skip_empty(labels_files[valid_idx].reshape(-1, N_CLASSES), probs.reshape(-1, N_CLASSES))
        fold_rows.append({
            'fold': fold,
            'valid_files': int(len(valid_idx)),
            'valid_sites': sorted(set(group_ids_all[valid_idx].tolist())),
            'best_valid_auc': float(max(history['valid_auc'])),
            'oof_auc': float(fold_auc),
        })

    oof_flat = oof_logits.reshape(-1, N_CLASSES)
    y_flat = labels_files.reshape(-1, N_CLASSES)
    overall_auc = macro_auc_skip_empty(y_flat, oof_flat)

    result = {
        'oof_probs': oof_logits,
        'oof_auc': float(overall_auc),
        'fold_summary': fold_rows,
    }
    return result


if RUN_OOF:
    oof_result = run_grouped_oof()
    oof_flat = oof_result['oof_probs'].reshape(-1, N_CLASSES)
    result_snapshot = {
        'experiment_id': CFG.experiment_id,
        'experiment_name': CFG.experiment_name,
        'n_splits': CFG.n_splits,
        'raw_perch_auc': raw_perch_auc,
        'legacy_exp003_best_auc': legacy_probe_auc,
        'oof_auc': oof_result['oof_auc'],
        'delta_vs_raw': float(oof_result['oof_auc'] - raw_perch_auc),
        'delta_vs_exp003': None if legacy_probe_auc is None else float(oof_result['oof_auc'] - legacy_probe_auc),
        'n_files': int(len(file_list)),
        'n_sites': int(len(set(group_ids_all.tolist()))),
        'model': {
            'd_model': CFG.d_model,
            'd_state': CFG.d_state,
            'n_ssm_layers': CFG.n_ssm_layers,
            'dropout': CFG.dropout,
        },
        'output_dir': str(OUTPUT_DIR),
    }
    save_json(result_snapshot, OUTPUT_DIR / 'result_snapshot.json')
    pd.DataFrame(oof_result['fold_summary']).to_csv(OUTPUT_DIR / 'fold_summary.csv', index=False)
    if CFG.save_oof_outputs:
        np.savez_compressed(OUTPUT_DIR / 'oof_outputs.npz', oof_probs=oof_result['oof_probs'].astype(np.float32), labels=labels_files.astype(np.float32))
        pd.DataFrame({'filename': file_list, 'site': group_ids_all, 'hour_utc': hours_all}).to_csv(OUTPUT_DIR / 'file_meta.csv', index=False)
    print(json.dumps(result_snapshot, indent=2))
else:
    print('OOF run skipped. Set RUN_OOF = True to train the temporal model.')



=== Fold 1 ===
{'epoch': 1, 'train_loss': 1.4235758781433105, 'valid_loss': 1.245254635810852, 'valid_auc': 0.8336697496673541}
{'epoch': 2, 'train_loss': 1.4188209772109985, 'valid_loss': 1.2311745882034302, 'valid_auc': 0.8335318565071719}
{'epoch': 3, 'train_loss': 1.3886713981628418, 'valid_loss': 1.206483244895935, 'valid_auc': 0.8333226726210926}
{'epoch': 4, 'train_loss': 1.3476344347000122, 'valid_loss': 1.1813238859176636, 'valid_auc': 0.8322259709056562}
{'epoch': 5, 'train_loss': 1.3229819536209106, 'valid_loss': 1.1553949117660522, 'valid_auc': 0.8310381699973735}
{'epoch': 6, 'train_loss': 1.3013923168182373, 'valid_loss': 1.1257914304733276, 'valid_auc': 0.8293788745350972}
{'epoch': 7, 'train_loss': 1.2807421684265137, 'valid_loss': 1.0920521020889282, 'valid_auc': 0.8284246221267337}
{'epoch': 8, 'train_loss': 1.2623485326766968, 'valid_loss': 1.0560216903686523, 'valid_auc': 0.8280530478684236}
{'epoch': 9, 'train_loss': 1.2456469535827637, 'valid_loss': 1.02091109752

In [10]:
def train_final_model() -> dict[str, tp.Any]:
    model, history = train_single_fold(
        emb_train=emb_files,
        logits_train=logits_files,
        labels_train=labels_files,
        site_train=site_ids_all,
        hour_train=hours_all,
        fam_train=file_families,
        emb_valid=emb_files,
        logits_valid=logits_files,
        labels_valid=labels_files,
        site_valid=site_ids_all,
        hour_valid=hours_all,
        fam_valid=file_families,
    )
    payload = {
        'model_state_dict': model.state_dict(),
        'cfg': asdict(CFG),
        'classes': PRIMARY_LABELS,
        'site_to_idx': site_to_idx,
        'n_families': n_families,
        'class_to_family': class_to_family,
    }
    torch.save(payload, OUTPUT_DIR / 'final_model.pt')
    summary = {
        'final_train_best_auc': float(max(history['valid_auc'])),
        'final_model_path': str(OUTPUT_DIR / 'final_model.pt'),
    }
    save_json(summary, OUTPUT_DIR / 'final_model_summary.json')
    return summary


if RUN_FINAL_TRAIN:
    final_summary = train_final_model()
    print(json.dumps(final_summary, indent=2))
else:
    print('Final train skipped. Set RUN_FINAL_TRAIN = True after OOF is satisfactory.')


Final train skipped. Set RUN_FINAL_TRAIN = True after OOF is satisfactory.
